# Chapter 3: Rule-Based Classifiers - Confusion Matrix & Classification Report

## Exercise 1: Execute the following code to classify Iris flowers using RIPPER algorithm

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report
import wittgenstein as lw

# 1. Load dataset
iris = load_iris(as_frame=True)
df = iris.data
df["class"] = iris.target
df["class"] = df["class"].map({0:"setosa", 1:"versicolor", 2:"virginica"})

X = df.drop(columns="class")
y = df["class"]

# 2. Huấn luyện RIPPER theo One-vs-Rest (OVR)
models = {}
for cls in y.unique():
    # chuyển True → 1, False → 0 → thành nhãn nhị phân [1, 0]
    y_binary = (y == cls).astype(int)  # one-vs-rest
    ripper = lw.RIPPER()
    ripper.fit(X, y_binary, pos_class=1)
    models[cls] = ripper
    print(f"=== Luật cho class {cls} ===")
    print(ripper.ruleset_)
    print()

# 3. Dự đoán bằng cách chọn class có độ tin cậy (confidence) cao nhất
def predict_ovr(X, models):
    """
    Hàm dự đoán nhãn cho dữ liệu X bằng phương pháp One-vs-Rest (OVR).

    Ý tưởng:
    - Mỗi class có một mô hình RIPPER riêng (huấn luyện nhị phân: class đó = 1, còn lại = 0).
    - Với mỗi mẫu dữ liệu (row):
        + Ta lấy xác suất dự đoán "p(y=1)" từ mô hình của từng class.
        + Tập hợp các xác suất này thành một dictionary scores: {class: xác suất}.
        + Chọn class có xác suất (confidence) cao nhất làm nhãn dự đoán cuối cùng.
    - Cách này gọi là "Dự đoán bằng cách chọn lớp có độ tin cậy cao nhất".
    """

    preds = []
    for _, row in X.iterrows():
        scores = {}
        for cls, model in models.items():
            # predict_proba([row]) trả về [p(y=0), p(y=1)]
            # Ta lấy phần tử [1] = xác suất mẫu thuộc class này (positive class)
            prob = model.predict_proba([row])[0][1]
            scores[cls] = prob

        # Lấy class có xác suất cao nhất
        pred_class = max(scores, key=scores.get)
        preds.append(pred_class)

    return preds

y_pred = predict_ovr(X, models)

# 4. Đánh giá
y_true = y.tolist()
acc = accuracy_score(y_true, y_pred)
print("\n=== Độ chính xác trên toàn bộ Iris dataset ===")
print(f"Accuracy = {acc:.3f}")

# 5. Thêm nhãn dự đoán vào dataframe
df["predicted"] = y_pred

# 6. Trực quan hóa true vs predicted
color_map = {"setosa":"red", "versicolor":"green", "virginica":"blue"}

plt.figure(figsize=(12,5))

# True labels
plt.subplot(1,2,1)
for cname, grp in df.groupby("class"):
    plt.scatter(grp["petal length (cm)"], grp["petal width (cm)"],
                label=cname, color=color_map[cname], edgecolor="k")
plt.title("True Labels")
plt.xlabel("petal length (cm)")
plt.ylabel("petal width (cm)")
plt.legend()

# Predicted labels
plt.subplot(1,2,2)
for pname, grp in df.groupby("predicted"):
    plt.scatter(grp["petal length (cm)"], grp["petal width (cm)"],
                label=pname, color=color_map[pname], edgecolor="k")
plt.title("Predicted Labels (RIPPER OVR)")
plt.xlabel("petal length (cm)")
plt.ylabel("petal width (cm)")
plt.legend()

plt.tight_layout()
plt.show()

# 7. Confusion Matrix
cm = confusion_matrix(y_true, y_pred, labels=["setosa","versicolor","virginica"])
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                              display_labels=["setosa","versicolor","virginica"])
disp.plot(cmap="Blues", values_format="d")
plt.title("Confusion Matrix (RIPPER OVR)")
plt.show()

# 8. Classification Report
print("\n=== Classification Report ===")
print(classification_report(y_true, y_pred, target_names=["setosa","versicolor","virginica"]))


## Exercise 2: Explain the confusion matrix

**Your answer:**
- ...

## Exercise 3: Explain the Classification Report

**Your answer:**
- ...
